# 🔍 Experiment 002: Group Fairness Analysis

Comprehensive fairness analysis for datasets and synthetic data generation.

| Parameter | Value |
|-----------|-------|
| **Experiment ID** | EXP-002 |
| **Name** | Group Fairness Analysis |
| **Sensitive Attrs** | Gender, Race |

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Imports successful")

## 1. Data Preparation

Create a sample dataset with known biases in the outcome variable.

In [ ]:
np.random.seed(42)
n = 10000

df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'gender': np.random.choice(['Male', 'Female'], n),
    'race': np.random.choice(['White', 'Black', 'Asian', 'Hispanic'], n, p=[0.4, 0.25, 0.2, 0.15]),
    'income': np.random.lognormal(10.5, 0.5, n),
    'credit_score': np.random.normal(700, 80, n).clip(300, 850),
})

df['outcome_prob'] = 0.3
df.loc[df['gender'] == 'Female', 'outcome_prob'] -= 0.08
df.loc[df['race'] == 'Black', 'outcome_prob'] -= 0.12
df['outcome_prob'] = df['outcome_prob'].clip(0.05, 0.95)
df['outcome'] = (np.random.random(n) < df['outcome_prob']).astype(int)
df = df.drop('outcome_prob', axis=1)

SENSITIVE_ATTRS = ['gender', 'race']
TARGET = 'outcome'

print(f"Dataset: {n} samples")
print(f"Sensitive: {SENSITIVE_ATTRS}")
print(f"Target: {TARGET}")

## 2. Group Fairness Analysis

Compute **Demographic Parity** metrics. DP Difference > 0.1 indicates significant disparity.

In [ ]:
print("⚖️ GROUP FAIRNESS ANALYSIS")
print("=" * 50)

for attr in SENSITIVE_ATTRS:
    print(f"\n【{attr.upper()}】")
    rates = df.groupby(attr)[TARGET].mean()
    
    for g, r in rates.items():
        print(f"  {g}: {r:.4f} ({r*100:.2f}%)")
    
    dp_diff = rates.max() - rates.min()
    dp_ratio = rates.min() / rates.max()
    
    print(f"\n  DP Difference: {dp_diff:.4f}")
    print(f"  DP Ratio: {dp_ratio:.4f}")
    
    if dp_diff > 0.1:
        print("  Status: ❌ Significant disparity")
    else:
        print("  Status: ✅ Acceptable")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, attr in enumerate(SENSITIVE_ATTRS):
    rates = df.groupby(attr)[TARGET].mean().sort_values(ascending=False)
    colors = ['#ff6b6b' if r < 0.3 else '#ffd93d' if r < 0.35 else '#4ecdc4' for r in rates]
    axes[idx].bar(range(len(rates)), rates.values, color=colors, edgecolor='black')
    axes[idx].set_xticks(range(len(rates)))
    axes[idx].set_xticklabels(rates.index, rotation=45)
    axes[idx].axhline(df[TARGET].mean(), color='red', linestyle='--', label='Overall')
    axes[idx].set_title(f'Selection Rate by {attr}', fontweight='bold')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## 3. Individual Fairness

Measure **consistency**: similar individuals should receive similar outcomes.

In [ ]:
print("\n👤 INDIVIDUAL FAIRNESS ANALYSIS")
print("=" * 50)

feature_cols = ['age', 'income', 'credit_score']
X = StandardScaler().fit_transform(df[feature_cols])
y = df[TARGET].values

sample_idx = np.random.choice(len(df), 1000, replace=False)
X_sample = X[sample_idx]
y_sample = y[sample_idx]

nn = NearestNeighbors(n_neighbors=6)
nn.fit(X_sample)
_, indices = nn.kneighbors(X_sample)

consistencies = []
for i in range(len(X_sample)):
    neighbor_labels = y_sample[indices[i, 1:]]
    consistencies.append(np.mean(neighbor_labels == y_sample[i]))

overall_consistency = np.mean(consistencies)
print(f"Consistency Score: {overall_consistency:.4f}")

plt.figure(figsize=(10, 5))
plt.hist(consistencies, bins=20, color='steelblue', edgecolor='black')
plt.axvline(overall_consistency, color='red', linestyle='--', label=f'Mean: {overall_consistency:.3f}')
plt.xlabel('Consistency Score')
plt.ylabel('Count')
plt.title('Individual Consistency Distribution', fontweight='bold')
plt.legend()
plt.show()

## 4. Intersectional Fairness

Analyze fairness across **intersectional groups** (e.g., Female + Black).

In [ ]:
print("\n🔗 INTERSECTIONAL FAIRNESS")
print("=" * 50)

df['intersectional'] = df['gender'] + '_' + df['race']
int_rates = df.groupby('intersectional')[TARGET].agg(['mean', 'count'])
int_rates.columns = ['rate', 'count']
int_rates = int_rates.sort_values('rate', ascending=False)
print(int_rates)

pivot = df.pivot_table(values=TARGET, index='gender', columns='race', aggfunc='mean')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=df[TARGET].mean())
plt.title('Intersectional Selection Rates', fontweight='bold')
plt.show()

## 5. Recommendations

In [ ]:
print("\n💡 RECOMMENDATIONS")
print("=" * 50)

recommendations = [
    "1. Apply adversarial debiasing for gender bias",
    "2. Use reweighing for racial disparity",
    "3. Monitor intersectional groups closely",
    "4. Consider post-processing calibration"
]

for r in recommendations:
    print(f"  {r}")